# 03 — Logistics network graph analytics

**Delivery ETA Intelligence** · Supply chain network intelligence

---

This notebook models the delivery network as a **directed weighted graph**: hubs are nodes, lanes (`source_center` → `destination_center`) are edges. We quantify **structural importance** (centrality), **congestion risk** (bottlenecks), and **lane criticality** (traffic × delay) to inform capacity planning and ETA modeling.

**Upstream:** `01_dataset_understanding`, `02_eda_time_and_routes`, processed features in `data/processed/`.

## Executive framing

| Question | Graph lens |
|----------|------------|
| Which hubs choke the network? | High betweenness + volume + delay |
| Where should we add capacity? | Top traffic edges with elevated delay ratios |
| Which nodes matter for ETA features? | PageRank / degree on the operational subgraph |
| Are there regional clusters? | Community structure (Louvain) |

Outputs feed the dashboard, bottleneck module (`src/bottleneck_analysis.py`), and graph embeddings (node2vec) in later ML work.

## 1. Environment setup

In [1]:
from __future__ import annotations

from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display

# Reproducibility & presentation
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

sns.set_theme(style="whitegrid", context="notebook", palette="deep")
plt.rcParams.update({
    "figure.figsize": (11, 6),
    "figure.dpi": 110,
    "axes.titlesize": 12,
    "axes.labelsize": 10,
})

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
FIGURES_DIR = PROJECT_ROOT / "outputs" / "figures"
TABLES_DIR = PROJECT_ROOT / "outputs" / "tables"

for d in (FIGURES_DIR, TABLES_DIR):
    d.mkdir(parents=True, exist_ok=True)

NOTEBOOK_TAG = "03"

### Helper utilities

Centralized functions keep the analysis reproducible and portable to `src/graph_construction.py` and `src/bottleneck_analysis.py`.

In [2]:
def resolve_data_path(directory: Path, stem: str) -> Path:
    """Prefer parquet; fall back to csv."""
    for ext in (".parquet", ".csv"):
        candidate = directory / f"{stem}{ext}"
        if candidate.exists():
            return candidate
    raise FileNotFoundError(f"No dataset '{stem}' in {directory}")


def load_logistics_table(path: Path) -> pd.DataFrame:
    if path.suffix == ".parquet":
        return pd.read_parquet(path)
    return pd.read_csv(path)


def top_n(df: pd.DataFrame, col: str, n: int = 10) -> pd.DataFrame:
    return df.nlargest(n, col).reset_index(drop=True)


def save_figure(fig: plt.Figure, name: str) -> Path:
    out = FIGURES_DIR / f"{NOTEBOOK_TAG}_{name}.png"
    fig.savefig(out, dpi=150, bbox_inches="tight")
    plt.close(fig)
    return out


def save_table(df: pd.DataFrame, name: str) -> Path:
    out = TABLES_DIR / f"{NOTEBOOK_TAG}_{name}.csv"
    df.to_csv(out, index=False)
    return out

## 2. Load processed logistics dataset

We consume the **cleaned segment-level table** produced after EDA and preprocessing (`delivery_logistics_processed`).  
Regenerate with: `python scripts/build_processed_dataset.py`

In [3]:
DATA_PATH = resolve_data_path(PROCESSED_DIR, "delivery_logistics_processed")
df = load_logistics_table(DATA_PATH)

required = {
    "source_center", "destination_center", "source_name", "destination_name",
    "actual_time", "osrm_time", "segment_actual_time", "segment_osrm_time",
    "segment_delay_ratio", "route_type", "route_lane",
}
missing = required - set(df.columns)
if missing:
    raise ValueError(f"Missing columns: {missing}")

print(f"Source: {DATA_PATH}")
print(f"Segments: {len(df):,} | Lanes: {df['route_lane'].nunique():,} | Hubs (unique centers): "
      f"{pd.unique(df[['source_center','destination_center']].values.ravel('K')).size:,}")

Source: D:\delivery eta\data\processed\delivery_logistics_processed.parquet
Segments: 142,502 | Lanes: 2,783 | Hubs (unique centers): 1,657


**Supply chain note:** Each row is an operational **segment** on a lane. Aggregating to edges yields a **lane-level flow network** suitable for hub capacity and corridor planning—not individual parcel GPS traces.

## 3. Build the directed logistics graph

- **Nodes:** `source_center` / `destination_center` (distribution centers, IPs, DPPs)
- **Edges:** directed `source → destination`
- **Primary weight:** segment count (traffic volume on the lane)
- **Edge attributes:** human-readable names, mean delay ratio, mean actual segment time

In [4]:
def aggregate_lane_edges(data: pd.DataFrame) -> pd.DataFrame:
    """Collapse segments to weighted directed edges."""
    edges = (
        data.groupby(
            ["source_center", "destination_center"],
            as_index=False,
        )
        .agg(
            weight=("route_lane", "count"),
            avg_delay=("segment_delay_ratio", "mean"),
            avg_actual_time=("segment_actual_time", "mean"),
            avg_osrm_time=("segment_osrm_time", "mean"),
            source_name=("source_name", "first"),
            destination_name=("destination_name", "first"),
            route_types=("route_type", lambda s: ",".join(sorted(s.dropna().unique())[:3])),
        )
    )
    edges["route_lane"] = (
        edges["source_center"].astype(str) + " → " + edges["destination_center"].astype(str)
    )
    return edges


def build_logistics_digraph(edge_table: pd.DataFrame) -> nx.DiGraph:
    """Construct NetworkX DiGraph with lane-level attributes."""
    G = nx.DiGraph()
    for row in edge_table.itertuples(index=False):
        G.add_edge(
            row.source_center,
            row.destination_center,
            weight=int(row.weight),
            avg_delay=float(row.avg_delay),
            avg_actual_time=float(row.avg_actual_time),
            avg_osrm_time=float(row.avg_osrm_time),
            source_name=row.source_name,
            destination_name=row.destination_name,
            route_lane=row.route_lane,
            route_types=row.route_types,
        )
    return G


def hub_traffic_volume(G: nx.DiGraph) -> pd.Series:
    """Total incident edge weight (in + out) per hub."""
    vol = {}
    for n in G.nodes():
        out_w = sum(d.get("weight", 1) for _, _, d in G.out_edges(n, data=True))
        in_w = sum(d.get("weight", 1) for _, _, d in G.in_edges(n, data=True))
        vol[n] = in_w + out_w
    return pd.Series(vol, name="traffic_volume")


edge_df = aggregate_lane_edges(df)
G = build_logistics_digraph(edge_df)
traffic = hub_traffic_volume(G)

for node, vol in traffic.items():
    G.nodes[node]["traffic_volume"] = int(vol)

print(f"Graph: {G.number_of_nodes():,} nodes, {G.number_of_edges():,} edges")
edge_df.head()

Graph: 1,657 nodes, 2,783 edges


,source_center,destination_center,weight,avg_delay,avg_actual_time,avg_osrm_time,source_name,destination_name,route_types,route_lane
0,IND000000AAL,IND411033AAA,35,3.069634,44.942857,15.314286,Pune_PC (Maharashtra),Pune_Tathawde_H (Maharashtra),Carting,IND000000AAL → IND411033AAA
1,IND000000AAQ,IND700028AAB,4,5.416667,42.250000,7.000000,Barasat_KrshnNgr_D (West Bengal),CCU_DumDum_DPC (West Bengal),Carting,IND000000AAQ → IND700028AAB
2,IND000000AAS,IND783370AAC,18,2.580453,30.500000,14.500000,Bongaigaon_Chpaguri_D (Assam),Kokrajhar_PigonDPP_D (Assam),FTL,IND000000AAS → IND783370AAC
3,IND000000AAZ,IND444203AAA,3,4.059096,95.666667,25.666667,Buldhana_Thsil3PL_D (Maharashtra),Shegaon_SChwkDPP_D (Maharashtra),FTL,IND000000AAZ → IND444203AAA
4,IND000000AAZ,IND444303AAA,2,2.266184,79.500000,34.000000,Buldhana_Thsil3PL_D (Maharashtra),Khamgaon_WamanDPP_D (Maharashtra),FTL,IND000000AAZ → IND444303AAA


**Logistics implication:** Edge weight proxies **flow intensity**. Lanes with high weight and high `avg_delay` are prime candidates for schedule buffers, alternate routing, or incremental capacity (vehicles, sortation shifts).

## 4. Global topology statistics

In [5]:
def graph_summary_stats(G: nx.DiGraph) -> pd.DataFrame:
    n, m = G.number_of_nodes(), G.number_of_edges()
    density = nx.density(G)
    avg_degree = float(np.mean([d for _, d in G.degree()])) if n else 0.0

    # Weak connectivity treats the graph as undirected for component analysis
    U = G.to_undirected()
    components = list(nx.connected_components(U))
    largest = max(components, key=len) if components else set()

    return pd.DataFrame([{
        "nodes": n,
        "edges": m,
        "density": round(density, 6),
        "avg_degree": round(avg_degree, 2),
        "weakly_connected_components": len(components),
        "largest_component_nodes": len(largest),
        "largest_component_pct": round(len(largest) / n * 100, 2) if n else 0,
        "is_dag": nx.is_directed_acyclic_graph(G),
    }])


summary_stats = graph_summary_stats(G)
summary_stats

,nodes,edges,density,avg_degree,weakly_connected_components,largest_component_nodes,largest_component_pct,is_dag
0,1657,2783,0.001014,3.36,64,1353,81.65,False


**Insight:** Logistics networks are typically **sparse** (density ≪ 1): each hub connects to a fraction of all hubs. Multiple weak components may indicate **regional silos** or data coverage gaps—worth validating with operations before assuming one national backbone.

## 5. Centrality analysis — structural hub importance

In [6]:
def compute_centrality_tables(G: nx.DiGraph, traffic: pd.Series) -> dict[str, pd.DataFrame]:
    """Degree, betweenness, closeness (on largest weak component), PageRank."""
    # Inverse volume as distance: high-traffic edges are "shorter" in flow terms
    for _, _, d in G.edges(data=True):
        d["flow_distance"] = 1.0 / max(d.get("weight", 1), 1)

    degree = nx.degree_centrality(G)
    between = nx.betweenness_centrality(G, weight="flow_distance", normalized=True)
    pagerank = nx.pagerank(G, weight="weight")

    # Closeness on largest undirected component (stable for disconnected networks)
    U = G.to_undirected()
    largest_nodes = max(nx.connected_components(U), key=len)
    sub = U.subgraph(largest_nodes).copy()
    close = nx.closeness_centrality(sub, distance="weight")
    close_full = {n: close.get(n, np.nan) for n in G.nodes()}

    def to_frame(scores: dict, metric: str) -> pd.DataFrame:
        out = pd.DataFrame({"hub_id": list(scores.keys()), metric: list(scores.values())})
        out = out.merge(
            traffic.rename("traffic_volume"),
            left_on="hub_id",
            right_index=True,
            how="left",
        )
        return out.sort_values(metric, ascending=False).reset_index(drop=True)

    return {
        "degree": to_frame(degree, "degree_centrality"),
        "betweenness": to_frame(between, "betweenness_centrality"),
        "closeness": to_frame(close_full, "closeness_centrality"),
        "pagerank": to_frame(pagerank, "pagerank"),
    }


centrality = compute_centrality_tables(G, traffic)
TOP_K = 10

In [7]:
for name, table in centrality.items():
    metric_col = [c for c in table.columns if c not in ("hub_id", "traffic_volume")][0]
    print(f"\n=== Top {TOP_K} hubs by {name} ===")
    display(top_n(table, metric_col, TOP_K))
    save_table(top_n(table, metric_col, 50), f"hub_rankings_{name}")


=== Top 10 hubs by degree ===


,hub_id,degree_centrality,traffic_volume
0,IND000000ACB,0.056763,38396
1,IND562132AAA,0.042874,20868
2,IND160002AAC,0.036836,5276
3,IND421302AAG,0.035024,14423
4,IND501359AAE,0.034420,8460
5,IND712311AAA,0.027778,7443
6,IND110037AAM,0.027174,5757
7,IND411033AAA,0.025966,7687
8,IND131028AAB,0.024155,4443
9,IND600056AAB,0.021739,3739



=== Top 10 hubs by betweenness ===


,hub_id,betweenness_centrality,traffic_volume
0,IND000000ACB,0.360461,38396
1,IND562132AAA,0.174206,20868
2,IND501359AAE,0.094087,8460
3,IND712311AAA,0.084577,7443
4,IND462022AAA,0.083188,3679
5,IND421302AAG,0.074881,14423
6,IND160002AAC,0.053204,5276
7,IND131028AAB,0.047053,4443
8,IND781018AAB,0.044012,2226
9,IND302014AAA,0.043919,2483



=== Top 10 hubs by closeness ===


,hub_id,closeness_centrality,traffic_volume
0,IND501359AAE,0.004174,8460
1,IND501218AAC,0.004140,13
2,IND000000ACB,0.004121,38396
3,IND501101AAB,0.004073,51
4,IND301705AAA,0.004071,23
5,IND122004AAA,0.004070,167
6,IND110037AAM,0.004069,5757
7,IND125033AAA,0.004062,25
8,IND505001AAB,0.004059,260
9,IND125001AAA,0.004058,407



=== Top 10 hubs by pagerank ===


,hub_id,pagerank,traffic_volume
0,IND000000ACB,0.043584,38396
1,IND562132AAA,0.027326,20868
2,IND501359AAE,0.018264,8460
3,IND421302AAG,0.015276,14423
4,IND712311AAA,0.014162,7443
5,IND209304AAA,0.011313,3737
6,IND411033AAA,0.010711,7687
7,IND110037AAM,0.010241,5757
8,IND160002AAC,0.010090,5276
9,IND751002AAB,0.008701,3982


| Metric | Operations interpretation |
|--------|---------------------------|
| **Degree** | Highly connected hubs (many lanes)—sortation / cross-dock load |
| **Betweenness** | Hubs on many shortest paths—**failure or delay here propagates network-wide** |
| **Closeness** | Can reach others quickly in the giant component—good repositioning anchors |
| **PageRank** | Influential in the **flow** sense (weighted by volume)—priority for monitoring |

These metrics become **graph features** for ETA models (e.g. origin betweenness, destination PageRank).

## 6. Bottleneck identification — congestion × structure

In [8]:
def minmax_scale(s: pd.Series) -> pd.Series:
    lo, hi = s.min(), s.max()
    if hi == lo:
        return pd.Series(0.0, index=s.index)
    return (s - lo) / (hi - lo)


def build_bottleneck_score(centrality_tables: dict[str, pd.DataFrame], G: nx.DiGraph) -> pd.DataFrame:
    """Composite score: betweenness + traffic + outgoing delay pressure."""
    base = centrality_tables["betweenness"][["hub_id", "betweenness_centrality", "traffic_volume"]].copy()

    out_delay = []
    for n in G.nodes():
        delays = [d.get("avg_delay", np.nan) for _, _, d in G.out_edges(n, data=True)]
        out_delay.append(float(np.nanmean(delays)) if delays else np.nan)
    base["mean_outgoing_delay"] = out_delay

    base["score_betweenness"] = minmax_scale(base["betweenness_centrality"])
    base["score_traffic"] = minmax_scale(base["traffic_volume"])
    base["score_delay"] = minmax_scale(
        base["mean_outgoing_delay"].fillna(base["mean_outgoing_delay"].median())
    )
    base["bottleneck_score"] = (
        0.45 * base["score_betweenness"]
        + 0.35 * base["score_traffic"]
        + 0.20 * base["score_delay"]
    )
    return base.sort_values("bottleneck_score", ascending=False).reset_index(drop=True)


bottlenecks = build_bottleneck_score(centrality, G)
bottleneck_top = bottlenecks.head(15)
display(bottleneck_top)
save_table(bottlenecks.head(100), "bottleneck_hubs")

,hub_id,betweenness_centrality,traffic_volume,mean_outgoing_delay,score_betweenness,score_traffic,score_delay,bottleneck_score
0,IND000000ACB,3.604613e-01,38396,3.069634,1.000000,1.000000,0.040830,0.808166
1,IND562132AAA,1.742057e-01,20868,2.319470,0.483285,0.543482,0.026888,0.413075
2,IND421302AAG,7.488105e-02,14423,3.803786,0.207737,0.375622,0.054474,0.235844
3,IND501359AAE,9.408687e-02,8460,5.416667,0.261018,0.220315,0.084449,0.211458
4,IND282001AAA,0.000000e+00,253,54.680181,0.000000,0.006563,1.000000,0.202297
5,IND712311AAA,8.457682e-02,7443,5.563325,0.234635,0.193827,0.087174,0.190860
6,IND742147AAA,1.614928e-03,38,43.903915,0.004480,0.000964,0.799726,0.162298
7,IND000000AFF,3.648730e-07,6,41.707062,0.000001,0.000130,0.758898,0.151826
8,IND462022AAA,8.318848e-02,3679,2.580453,0.230783,0.095794,0.031738,0.143728
9,IND532201AAA,1.212838e-03,75,36.430161,0.003365,0.001927,0.660828,0.134354


WindowsPath('D:/delivery eta/outputs/tables/03_bottleneck_hubs.csv')

### Why these hubs are operational bottlenecks

A hub ranks high when it combines:

1. **Structural chokepoint** — high betweenness: many efficient paths flow through it; outages or dwell time spikes create cascading delays.
2. **Volume stress** — high traffic: sortation, yard, and dock constraints bind at scale.
3. **Service degradation** — elevated outgoing delay ratios: the router underestimates true time on departures from this node.

**Mitigation levers:** duplicate processing windows, overflow facilities, dynamic cutoff rules, and targeted OSRM calibration on incident edges.

In [9]:
fig, ax = plt.subplots(figsize=(10, 6))
plot_df = bottleneck_top.head(12)
sns.barplot(data=plot_df, y="hub_id", x="bottleneck_score", ax=ax, hue="hub_id", legend=False)
ax.set_title("Top bottleneck hubs (composite structural + volume + delay score)")
ax.set_xlabel("Bottleneck score (0–1)")
save_figure(fig, "bottleneck_hubs_bar")

WindowsPath('D:/delivery eta/outputs/figures/03_bottleneck_hubs_bar.png')

## 7. Route importance — volume, delay, and criticality

In [10]:
routes = edge_df.copy()
routes["criticality"] = routes["weight"] * routes["avg_delay"]

busiest = top_n(routes, "weight", 15)
highest_delay = top_n(routes, "avg_delay", 15)
most_critical = top_n(routes, "criticality", 15)

route_exports = pd.concat([
    busiest.assign(rank_type="busiest"),
    highest_delay.assign(rank_type="highest_delay"),
    most_critical.assign(rank_type="critical"),
], ignore_index=True)
save_table(route_exports, "route_importance")

display(busiest.head(10))

,source_center,destination_center,weight,avg_delay,avg_actual_time,avg_osrm_time,source_name,destination_name,route_types,route_lane,criticality
0,IND000000ACB,IND562132AAA,4970,1.776631,38.032797,23.094366,Gurgaon_Bilaspur_HB (Haryana),Bangalore_Nelmngla_H (Karnataka),FTL,IND000000ACB → IND562132AAA,8829.856794
1,IND562132AAA,IND000000ACB,3316,1.705791,36.708082,23.709891,Bangalore_Nelmngla_H (Karnataka),Gurgaon_Bilaspur_HB (Haryana),FTL,IND562132AAA → IND000000ACB,5656.401620
2,IND000000ACB,IND712311AAA,2831,2.077405,36.416107,19.206641,Gurgaon_Bilaspur_HB (Haryana),Kolkata_Dankuni_HB (West Bengal),FTL,IND000000ACB → IND712311AAA,5881.133979
3,IND000000ACB,IND501359AAE,1638,1.966965,40.447497,22.666667,Gurgaon_Bilaspur_HB (Haryana),Hyderabad_Shamshbd_H (Telangana),FTL,IND000000ACB → IND501359AAE,3221.888939
4,IND000000ACB,IND421302AAG,1616,1.846625,38.403465,21.643564,Gurgaon_Bilaspur_HB (Haryana),Bhiwandi_Mankoli_HB (Maharashtra),FTL,IND000000ACB → IND421302AAG,2984.146783
5,IND421302AAG,IND000000ACB,1268,1.922581,39.734227,21.429022,Bhiwandi_Mankoli_HB (Maharashtra),Gurgaon_Bilaspur_HB (Haryana),FTL,IND421302AAG → IND000000ACB,2437.832815
6,IND781018AAB,IND110037AAM,1137,2.106779,49.964820,25.084433,Guwahati_Hub (Assam),Delhi_Airport_H (Delhi),FTL,IND781018AAB → IND110037AAM,2395.407249
7,IND421302AAG,IND562132AAA,1129,1.863416,35.558016,20.296723,Bhiwandi_Mankoli_HB (Maharashtra),Bangalore_Nelmngla_H (Karnataka),FTL,IND421302AAG → IND562132AAA,2103.797001
8,IND000000ACB,IND411033AAA,1120,1.627478,34.704464,22.745536,Gurgaon_Bilaspur_HB (Haryana),Pune_Tathawde_H (Maharashtra),FTL,IND000000ACB → IND411033AAA,1822.775325
9,IND000000ACB,IND600056AAB,1014,1.835140,42.674556,24.397436,Gurgaon_Bilaspur_HB (Haryana),MAA_Poonamallee_HB (Tamil Nadu),FTL,IND000000ACB → IND600056AAB,1860.832282


In [11]:
def plot_route_rank(data: pd.DataFrame, y_col: str, title: str, fname: str, label_col: str = "route_lane"):
    fig, ax = plt.subplots(figsize=(11, 6))
    d = data.head(12).copy()
    sns.barplot(data=d, y=label_col, x=y_col, ax=ax, hue=label_col, legend=False)
    ax.set_title(title)
    plt.tight_layout()
    save_figure(fig, fname)


plot_route_rank(busiest, "weight", "Busiest lanes (segment count)", "routes_busiest")
plot_route_rank(highest_delay, "avg_delay", "Highest mean segment delay ratio", "routes_highest_delay")
plot_route_rank(most_critical, "criticality", "Most critical lanes (volume × delay)", "routes_critical")

**Supply chain optimization:** Prioritize **critical** lanes (not just busiest) for ETA model retraining, driver allocation, and hub SLA reviews. A medium-volume lane with extreme delay can hurt OTP more than a high-volume stable corridor.

## 8. Network visualization (top-N subgraph)

Full graphs (~1.6k nodes) are unreadable. We render the **induced subgraph** on the top hubs by traffic and their strongest incident edges.

In [12]:
TOP_HUBS_VIZ = 35
TOP_EDGES_VIZ = 80


def extract_top_subgraph(G: nx.DiGraph, top_hubs: int, top_edges: int) -> nx.DiGraph:
    hub_rank = hub_traffic_volume(G).nlargest(top_hubs).index.tolist()
    edge_rank = sorted(G.edges(data=True), key=lambda e: e[2].get("weight", 0), reverse=True)[:top_edges]
    nodes = set(hub_rank)
    for u, v, _ in edge_rank:
        nodes.add(u)
        nodes.add(v)
    sub_edges = [(u, v) for u, v, d in G.edges(data=True) if u in nodes and v in nodes]
    return G.edge_subgraph(sub_edges).copy()


def draw_logistics_network(H: nx.DiGraph, traffic: pd.Series, seed: int = 42) -> plt.Figure:
    fig, ax = plt.subplots(figsize=(14, 10))
    pos = nx.spring_layout(H, seed=seed, k=0.45, iterations=60)

    node_sizes = [300 + 8 * traffic.get(n, 0) for n in H.nodes()]
    edge_widths = [0.5 + 0.04 * d.get("weight", 1) for _, _, d in H.edges(data=True)]

    nx.draw_networkx_nodes(H, pos, node_size=node_sizes, node_color="#4C72B0", alpha=0.85, ax=ax)
    nx.draw_networkx_edges(
        H, pos, width=edge_widths, edge_color="#555555", alpha=0.45,
        arrows=True, arrowsize=12, connectionstyle="arc3,rad=0.08", ax=ax,
    )
    # Label only highest-traffic nodes in the subgraph
    label_nodes = traffic.loc[list(H.nodes())].nlargest(12).index
    labels = {n: n[:12] for n in label_nodes}
    nx.draw_networkx_labels(H, pos, labels=labels, font_size=8, ax=ax)

    ax.set_title(
    "Logistics network (top hubs & lanes) — "
    "node size proportional to traffic, edge width proportional to volume"
)
    ax.axis("off")
    return fig


H = extract_top_subgraph(G, TOP_HUBS_VIZ, TOP_EDGES_VIZ)
fig_net = draw_logistics_network(H, traffic)
save_figure(fig_net, "network_topn")

WindowsPath('D:/delivery eta/outputs/figures/03_network_topn.png')

Use this view in steering committees to **anchor qualitative discussions** (which regions look over-connected) before drilling into lane-level CSV exports.

## 9. Community detection — regional cluster structure

In [13]:
def run_louvain_communities(G: nx.DiGraph, seed: int = 42) -> pd.DataFrame:
    """Louvain on undirected projection; weight = lane volume."""

    U = nx.Graph()

    for u, v, d in G.edges(data=True):
        w = d.get("weight", 1)

        if U.has_edge(u, v):
            U[u][v]["weight"] += w
        else:
            U.add_edge(u, v, weight=w)

    communities = nx.community.louvain_communities(
        U,
        weight="weight",
        seed=seed
    )

    rows = []

    for cid, members in enumerate(communities):
        for hub in members:
            rows.append({
                "hub_id": hub,
                "community_id": cid,
                "community_size": len(members)
            })

    return pd.DataFrame(rows)


# Run community detection
communities_df = run_louvain_communities(G, RANDOM_STATE)

# Community summary
comm_summary = (
    communities_df
    .groupby("community_id")
    .agg(
        hubs=("hub_id", "count"),
        sample_hubs=("hub_id",
                     lambda s: ", ".join(list(s.head(5))))
    )
    .sort_values("hubs", ascending=False)
    .reset_index()
)

# Display results
display(comm_summary.head(10))

# Save output
save_table(communities_df, "hub_communities")

,community_id,hubs,sample_hubs
0,83,160,"IND602105AAB, IND506169AAB, IND506132AAA, IND5..."
1,8,160,"IND110020AAB, IND425109AAA, IND444203AAA, IND4..."
2,12,157,"IND251309AAB, IND700028AAB, IND743270AAA, IND1..."
3,7,83,"IND151103AAA, IND181152AAA, IND135001AAA, IND1..."
4,26,83,"IND679303AAA, IND563131AAA, IND560300AAA, IND6..."
5,11,82,"IND388121AAA, IND380021AAA, IND345001AAA, IND3..."
6,10,77,"IND272001AAA, IND208019AAA, IND282001AAA, IND2..."
7,2,37,"IND509124AAA, IND518401AAA, IND636705AAB, IND5..."
8,108,29,"IND741502AAB, IND723202AAA, IND713205AAB, IND7..."
9,98,28,"IND626607AAB, IND625008AAA, IND625106AAA, IND6..."


WindowsPath('D:/delivery eta/outputs/tables/03_hub_communities.csv')

**Logistics implication:** Communities approximate **regional sub-networks** (e.g. state clusters). Planning cutoffs, linehaul schedules, and inventory positioning per community reduces long-range dependency on national backbone hubs.

In [14]:
fig, ax = plt.subplots(figsize=(10, 5))
sns.histplot(comm_summary["hubs"], bins=30, ax=ax)
ax.set_title("Distribution of community sizes (Louvain)")
ax.set_xlabel("Hubs per community")
save_figure(fig, "community_size_distribution")

WindowsPath('D:/delivery eta/outputs/figures/03_community_size_distribution.png')

## 10. Centrality vs delay — diagnostic scatter

In [15]:
diag = bottlenecks.merge(centrality["pagerank"][["hub_id", "pagerank"]], on="hub_id", how="left")

fig, ax = plt.subplots(figsize=(9, 6))
sns.scatterplot(
    data=diag,
    x="traffic_volume",
    y="betweenness_centrality",
    size="mean_outgoing_delay",
    sizes=(40, 400),
    alpha=0.6,
    ax=ax,
)
ax.set_title(
    "Hub diagnostics: traffic vs betweenness "
    "(point size proportional to outgoing delay)"
)
ax.set_xlabel("Traffic volume (incident edge weight)")
ax.set_ylabel("Betweenness centrality")
save_figure(fig, "hub_scatter_diagnostics")

WindowsPath('D:/delivery eta/outputs/figures/03_hub_scatter_diagnostics.png')

## 11. Executive summary & recommendations

In [16]:
def hub_label(hub_id: str) -> str:
    return str(hub_id)


key_bottlenecks = bottlenecks.head(5)["hub_id"].tolist()
key_routes = most_critical.head(5)["route_lane"].tolist()

print("=" * 60)
print("DELIVERY ETA INTELLIGENCE — GRAPH ANALYTICS SUMMARY")
print("=" * 60)
print(summary_stats.to_string(index=False))
print()
print("Top bottleneck hubs:", ", ".join(key_bottlenecks))
print("Top critical routes:")
for r in key_routes:
    print(f"  • {r}")
print()
print("Artifacts:")
print(f"  Figures → {FIGURES_DIR}")
print(f"  Tables  → {TABLES_DIR}")

DELIVERY ETA INTELLIGENCE — GRAPH ANALYTICS SUMMARY
 nodes  edges  density  avg_degree  weakly_connected_components  largest_component_nodes  largest_component_pct  is_dag
  1657   2783 0.001014        3.36                           64                     1353                  81.65   False

Top bottleneck hubs: IND000000ACB, IND562132AAA, IND421302AAG, IND501359AAE, IND282001AAA
Top critical routes:
  • IND000000ACB → IND562132AAA
  • IND000000ACB → IND712311AAA
  • IND562132AAA → IND000000ACB
  • IND000000ACB → IND501359AAE
  • IND000000ACB → IND421302AAG

Artifacts:
  Figures → D:\delivery eta\outputs\figures
  Tables  → D:\delivery eta\outputs\tables
